# Discovery: `university.students`

Primer ejemplo del patron que vamos a repetir para las 18 tablas: leer la tabla cruda de `bronze`, perfilarla con pandas (nulos, duplicados, tipos, rangos raros), y a partir de eso decidir las reglas de limpieza que van a `sql/silver/university.sql`.

Esta notebook **no escribe en silver** — solo explora y documenta el razonamiento. La carga real a `silver.university__students` la hace el script `src/transform/build_silver.py`, ejecutando el SQL que definimos al final de este notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.university__students", engine)
df.shape

## 1. Forma general

Todo llega como `TEXT` desde bronze (a proposito, ver `docs/decisiones.md`). Miramos columnas, tipos actuales y una muestra.

In [ ]:
print(df.dtypes)
df.head()

## 2. Nulos y duplicados

In [ ]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("student_id duplicados:", df["student_id"].duplicated().sum())
print("Filas 100% duplicadas:", df.duplicated().sum())

## 3. Rangos y valores raros

- `country`: cuantos valores distintos hay, y si alguno se ve raro (typo, casing inconsistente).
- Fechas: que `birth_date` y `enrolled_at` sean fechas validas, y que `birth_date` sea siempre anterior a `enrolled_at` (nadie se inscribe antes de nacer).

In [ ]:
print("Valores distintos de country:")
print(df["country"].value_counts())
print()

birth = pd.to_datetime(df["birth_date"])
enrolled = pd.to_datetime(df["enrolled_at"])

print("Fechas invalidas (no parsean):", birth.isna().sum() + enrolled.isna().sum())
print("Filas con birth_date >= enrolled_at (inconsistente):", (birth >= enrolled).sum())

## 4. Conclusion: reglas de limpieza para silver

`students` resulta ser una tabla limpia (sin nulos, sin duplicados, sin fechas invertidas, `country` con 8 codigos ISO consistentes). No hace falta descartar ni corregir filas. Las reglas para `sql/silver/university.sql` son solo de **tipado y estandarizacion**, no de correccion de errores:

- `student_id` -> `TEXT PRIMARY KEY` (ya viene unico).
- `first_name`, `last_name` -> `TRIM`.
- `email` -> `TRIM` + `LOWER` (estandarizar mayusculas/minusculas para poder hacer joins/comparaciones confiables despues).
- `birth_date`, `enrolled_at` -> castear de texto a `DATE`.
- `country` -> `TRIM` + `UPPER` (ya vienen consistentes, pero se fuerza el estandar por si acaso).

Esto se implementa en `sql/silver/university.sql` y se ejecuta con `src/transform/build_silver.py university`.